In [3]:
#### ------------------------------------------------------------------------------------------
#### author: Ranjan Barman, date: may 30, 2025 (modified for immune only)
#### Compute 12 NPIFs based on HoverNet prediction, using MPP = 0.248 for unit conversion
#### Removes outliers for Major Axis and Minor Axis using IQR
#### Filters top 25% tiles based on immune nuclei count
#### Input: features3/
#### Output: NPIFs summary CSV across slides
#### ------------------------------------------------------------------------------------------

import os
import pandas as pd
import numpy as np

# Set working directory
_wpath_ = "/data/Lab_ruppin/Ranjan/HnE/"
os.makedirs(_wpath_, exist_ok=True)
os.chdir(_wpath_)
print("Working directory:", _wpath_)

# Dataset and output paths
dataset_name = "TCGA_BRCA_FFPE"
input_folder = f"{dataset_name}/outputs/HoverNet/"
output_file_path = f"{dataset_name}/outputs/HoverNet/HoverNet_NPIFs_TCGA_BRCA_1106_Filter_Top_25Q_ImmuneOnly.csv"

# Columns to compute NPIFs
columns_to_compute = ["Area", "Major Axis", "Minor Axis", "Perimeter", "Eccentricity", "Circularity"]

# Microns per pixel for scaling
MPP = 0.248

# Function to remove outliers using IQR
def remove_outliers(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 3 * IQR
    upper_bound = Q3 + 3 * IQR
    return df[(df[column] >= lower_bound) & (df[column] <= upper_bound)]

# Initialize result list
results = []

# Get all TCGA slide folders
tcga_folders = [
    f for f in os.listdir(input_folder)
    if os.path.isdir(os.path.join(input_folder, f)) and f.startswith("TCGA")
]

# Process each slide
for slide_name in tcga_folders:
    file_path = os.path.join(input_folder, slide_name, "features3", f"{slide_name}.csv")

    if not os.path.exists(file_path):
        print(f"File not found: {file_path}, skipping...")
        continue

    df = pd.read_csv(file_path)

    # Check for required columns
    if "Nucleus Type" not in df.columns or "Tile" not in df.columns:
        print(f"Required columns missing in {slide_name}, skipping...")
        continue

    # Keep only immune nuclei
    df = df[df["Nucleus Type"] == "Immune"]
    if df.empty:
        print(f"  - No immune nuclei found in {slide_name}, skipping...\n")
        continue

    # Convert units using MPP
    df["Area"] = df["Area"] * (MPP ** 2)
    df["Major Axis"] = df["Major Axis"] * MPP
    df["Minor Axis"] = df["Minor Axis"] * MPP
    df["Perimeter"] = df["Perimeter"] * MPP

    # Remove outliers
    df = remove_outliers(df, "Major Axis")
    df = remove_outliers(df, "Minor Axis")
    if df.empty:
        print(f"  - All immune nuclei removed as outliers in {slide_name}, skipping...\n")
        continue

    # Count immune nuclei per tile
    tile_counts = df.groupby("Tile")["Nucleus ID"].count().reset_index()
    tile_counts.rename(columns={"Nucleus ID": "Nucleus_Count"}, inplace=True)

    # Compute 75th percentile of immune nucleus counts
    total_tiles_with_immune = tile_counts.shape[0]
    percentile_75 = tile_counts["Nucleus_Count"].quantile(0.75)
    top_25_percent_tiles = tile_counts[tile_counts["Nucleus_Count"] >= percentile_75]
    num_filtered_tiles = top_25_percent_tiles.shape[0]

    print(f"Slide: {slide_name}")
    print(f"  - Total tiles with immune nuclei: {total_tiles_with_immune}")
    print(f"  - 75th percentile immune nucleus count threshold: {percentile_75}")
    print(f"  - Top 25% tile count: {num_filtered_tiles}")

    # Filter to top 25% immune-dense tiles
    df_filtered = df[df["Tile"].isin(top_25_percent_tiles["Tile"])]
    if df_filtered.empty:
        print(f"  - No valid top 25% immune tiles in {slide_name}, skipping...\n")
        continue

    # Compute NPIF statistics
    mean_values = df_filtered[columns_to_compute].mean()
    std_values = df_filtered[columns_to_compute].std()

    # Append results
    results.append(
        [slide_name, total_tiles_with_immune, num_filtered_tiles] +
        mean_values.tolist() +
        std_values.tolist()
    )

# Build final DataFrame
result_df = pd.DataFrame(
    results,
    columns=["Slide_Name", "Total_Tiles", "Filtered_Tiles"] +
            [f"Mean {col}" for col in columns_to_compute] +
            [f"Std {col}" for col in columns_to_compute]
)

# Handle NaNs and Infs
result_df.replace([np.inf, -np.inf], np.nan, inplace=True)
result_df.fillna(result_df.mean(numeric_only=True), inplace=True)
result_df.fillna(0, inplace=True)

# Save result
os.makedirs(os.path.dirname(output_file_path), exist_ok=True)
result_df.to_csv(output_file_path, index=False)

print(f"\nImmune-only NPIFs from top 25% tiles saved to: {output_file_path}")


Working directory: /data/Lab_ruppin/Ranjan/HnE/
Slide: TCGA-D8-A13Z-01Z-00-DX1_3624_tiles
  - Total tiles with immune nuclei: 2925
  - 75th percentile immune nucleus count threshold: 15.0
  - Top 25% tile count: 760
Slide: TCGA-AR-A0TR-01Z-00-DX1_3099_tiles
  - Total tiles with immune nuclei: 2698
  - 75th percentile immune nucleus count threshold: 12.0
  - Top 25% tile count: 727
Slide: TCGA-D8-A1JN-01Z-00-DX1_4957_tiles
  - Total tiles with immune nuclei: 4833
  - 75th percentile immune nucleus count threshold: 17.0
  - Top 25% tile count: 1259
Slide: TCGA-A2-A0ES-01Z-00-DX1_4815_tiles
  - Total tiles with immune nuclei: 4688
  - 75th percentile immune nucleus count threshold: 23.0
  - Top 25% tile count: 1235
Slide: TCGA-C8-A12U-01Z-00-DX1_4200_tiles
  - Total tiles with immune nuclei: 3768
  - 75th percentile immune nucleus count threshold: 23.0
  - Top 25% tile count: 950
Slide: TCGA-D8-A1JU-01Z-00-DX1_1890_tiles
  - Total tiles with immune nuclei: 1642
  - 75th percentile immune 

In [4]:
result_df

,Slide_Name,Total_Tiles,Filtered_Tiles,Mean Area,Mean Major Axis,Mean Minor Axis,Mean Perimeter,Mean Eccentricity,Mean Circularity,Std Area,Std Major Axis,Std Minor Axis,Std Perimeter,Std Eccentricity,Std Circularity
0,TCGA-D8-A13Z-01Z-00-DX1_3624_tiles,2925,760,4.912181,3.012063,2.249605,8.675528,0.614028,0.812938,1.450872,0.587625,0.319362,1.482220,0.155955,0.072400
1,TCGA-AR-A0TR-01Z-00-DX1_3099_tiles,2698,727,4.620095,2.925481,2.171165,8.375217,0.620032,0.813782,1.747505,0.657685,0.359560,1.707252,0.152263,0.071622
2,TCGA-D8-A1JN-01Z-00-DX1_4957_tiles,4833,1259,5.041088,2.964404,2.327670,8.653582,0.571534,0.834099,1.572618,0.534616,0.333140,1.387128,0.154867,0.058276
3,TCGA-A2-A0ES-01Z-00-DX1_4815_tiles,4688,1235,5.384223,3.165048,2.338663,9.024876,0.629180,0.821020,1.637952,0.603521,0.343456,1.515086,0.147706,0.063386
4,TCGA-C8-A12U-01Z-00-DX1_4200_tiles,3768,950,4.639509,2.929748,2.178321,8.382203,0.618213,0.814795,1.718553,0.643574,0.363612,1.649582,0.154094,0.069576
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1101,TCGA-BH-A18S-01Z-00-DX1_1472_tiles,828,226,6.045456,3.352457,2.454960,9.607328,0.636203,0.807345,2.191066,0.708165,0.430504,1.857831,0.147719,0.069640
1102,TCGA-OK-A5Q2-01Z-00-DX2_4517_tiles,4309,1165,5.243211,3.072271,2.324480,8.874889,0.601180,0.817784,2.136323,0.722071,0.420887,1.906754,0.155751,0.069426
1103,TCGA-EW-A1IX-01Z-00-DX1_2878_tiles,1719,507,4.941830,3.039397,2.229620,8.652373,0.634821,0.813533,1.855747,0.663081,0.388811,1.725871,0.145567,0.069403
1104,TCGA-B6-A0IO-01Z-00-DX1_2044_tiles,1868,532,5.660117,3.243193,2.380593,9.277773,0.633641,0.810061,2.084589,0.693042,0.411095,1.805819,0.148854,0.068480
